# Bhojpuri QLoRA Fine-Tune — Llama 3.1 8B Instruct

**Goal:** Fine-tune Llama 3.1 8B Instruct on 498 verified Bhojpuri Q&A examples to reduce Hindi drift.

**Runtime required:** L4 GPU (Colab Pro) — 24 GB VRAM

**Before running:**
1. Set `HF_TOKEN` in *Secrets* (key icon in the left sidebar).
2. Upload `eval-monitoring/finetune_corpus_v1.jsonl` when Cell 4 prompts you.
3. Run all cells top-to-bottom.

**Estimated time:** ~15–20 min on L4.

In [ ]:
# ── Cell 1: Verify GPU ────────────────────────────────────────────────────
import subprocess
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)

In [ ]:
# ── Cell 2: Install packages ──────────────────────────────────────────────
%%capture
!pip install -q \
  "transformers>=4.43.0" \
  "peft>=0.11.0" \
  "trl>=0.9.0" \
  "bitsandbytes>=0.43.0" \
  "accelerate>=0.30.0" \
  datasets
print('Install done')

In [ ]:
# ── Cell 3: HuggingFace login ─────────────────────────────────────────────
import os
from google.colab import userdata
from huggingface_hub import login

HF_TOKEN = userdata.get('HF_TOKEN')
os.environ['HF_TOKEN'] = HF_TOKEN
login(token=HF_TOKEN, add_to_git_credential=False)
print('HF login OK')

In [ ]:
# ── Cell 4: Upload dataset ────────────────────────────────────────────────
# Upload eval-monitoring/finetune_corpus_v1.jsonl from your local machine.
import os, json
from google.colab import files

DATASET_PATH = 'finetune_corpus_v1.jsonl'

if not os.path.exists(DATASET_PATH):
    print('Upload finetune_corpus_v1.jsonl ...')
    uploaded = files.upload()
    for fname in uploaded:
        os.rename(fname, DATASET_PATH)

with open(DATASET_PATH) as f:
    n = sum(1 for _ in f)
print(f'Dataset ready: {n} examples')

In [ ]:
# ── Cell 5: Load base model in 4-bit (QLoRA) ──────────────────────────────
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = 'meta-llama/Llama-3.1-8B-Instruct'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_TOKEN)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
    token=HF_TOKEN,
    torch_dtype=torch.bfloat16,
)
model.config.use_cache = False
model.config.pretraining_tp = 1

print(f'Model loaded on: {next(model.parameters()).device}')
print(f'VRAM used: {torch.cuda.memory_allocated()/1e9:.2f} GB')

In [ ]:
# ── Cell 6: Apply LoRA adapters ───────────────────────────────────────────
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        'q_proj', 'k_proj', 'v_proj', 'o_proj',
        'gate_proj', 'up_proj', 'down_proj',
    ],
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# ── Cell 7: Format dataset with Llama 3.1 chat template ───────────────────
import json
from datasets import Dataset

with open(DATASET_PATH) as f:
    raw = [json.loads(line) for line in f]

dataset = Dataset.from_list(raw)

def format_example(example):
    text = tokenizer.apply_chat_template(
        example['messages'],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {'text': text}

dataset = dataset.map(format_example, remove_columns=['messages'])
print(f'Formatted {len(dataset)} examples')
print('\nSample (truncated):')
print(dataset[0]['text'][:400])

In [ ]:
# ── Cell 8: Train ─────────────────────────────────────────────────────────
from trl import SFTTrainer, SFTConfig

OUTPUT_DIR = './bhojpuri-qlora-checkpoints'

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,

    # — epochs & batch —
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,   # effective batch size = 8
    gradient_checkpointing=True,

    # — optimiser —
    optim='paged_adamw_8bit',
    learning_rate=2e-4,
    lr_scheduler_type='cosine',
    warmup_steps=6,
    max_grad_norm=0.3,

    # — precision —
    bf16=True,
    fp16=False,

    # — logging & saving —
    logging_steps=10,
    save_strategy='epoch',
    report_to='none',

    # — sequence —
    dataset_text_field='text',
    packing=False,
)

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=dataset,
    args=training_args,
)

print('Starting training ...')
result = trainer.train()
t = result.metrics
print(f'\nDone in {t["train_runtime"]:.0f}s  '
      f'({t["train_samples_per_second"]:.1f} samples/s)  '
      f'loss={t["train_loss"]:.4f}')

In [ ]:
# ── Cell 9: Save LoRA adapter ─────────────────────────────────────────────
import shutil

ADAPTER_DIR = './bhojpuri-lora-adapter'
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f'Adapter saved to {ADAPTER_DIR}')

# ── Copy to Google Drive so it survives runtime disconnect ────────────────
try:
    from google.colab import drive
    drive.mount('/content/drive')
    drive_dest = '/content/drive/MyDrive/bhojpuri-lora-adapter'
    if os.path.exists(drive_dest):
        shutil.rmtree(drive_dest)
    shutil.copytree(ADAPTER_DIR, drive_dest)
    print(f'Adapter also saved to Google Drive: {drive_dest}')
except Exception as e:
    print(f'Drive copy skipped ({e}) — download manually if needed.')

In [ ]:
# ── Cell 10: Quick inference test ─────────────────────────────────────────
# Exit training mode: gradient checkpointing keeps use_cache=False otherwise
model.gradient_checkpointing_disable()
model.config.use_cache = True
model.eval()

# Production system prompt — must match build_bhojpuri_messages() in app.py
SYSTEM_PROMPT = (
    'तू एगो भोजपुरी सहायक बाड़s। '
    'हर जवाब सुद्ध भोजपुरी में दे — हिंदी में बिल्कुल ना। '
    'जवाब 1 से 3 छोट वाक्य में दे। '
    'बेकार भूमिका मत लिख। '
    'एकही बात बार-बार मत दोहरा। '
    'अंग्रेजी शब्द कम से कम रख। '
    "जवाब में 'जवाब:' मत लिख। "
    'खेती से जुड़ल सवाल में सीधा, छोट आ काम के सलाह दे। '
    'लाइव जानकारी जइसे मौसम, बाजार भाव, खबर न होखे त साफ कह दे — अंदाजा मत लगा।'
)

def ask(question):
    msgs = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user',   'content': question},
    ]
    prompt = tokenizer.apply_chat_template(
        msgs, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=150,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(
        out[0][inputs['input_ids'].shape[1]:],
        skip_special_tokens=True,
    ).strip()

# Tests cover known failure modes from eval: proper nouns, ag, refusal, health, scheme
tests = [
    'बिहार के मुख्यमंत्री कवन बाड़न?',       # proper noun (was 'कुंवर' bug)
    'धान के पत्ता पीयर काहें हो जाला?',       # agriculture
    'आजु बाजार में सरसों के भाव का बा?',      # refusal — no live data
    'बुखार में का करे के चाही?',               # health
    'पीएम किसान योजना का हऊ?',                # government scheme
    'सूरज कवन दिशा में उगेला?',               # general knowledge
]

print('=' * 62)
print('POST-FINETUNE INFERENCE TEST')
print('=' * 62)
for q in tests:
    print(f'\nQ: {q}')
    print(f'A: {ask(q)}')
print('\n' + '=' * 62)

## What to check in the output above

| Test | Pass criterion |
|---|---|
| Bihar CM | Answer contains **नीतीश कुमार** (not कुंवर) |
| Yellow paddy leaves | Answer uses **बा / बाड़न / बानी** — no **है / हैं** |
| Mustard price | Model says **नइखे** and redirects to mandi |
| Fever | Imperative ends in **-ऽ** (करऽ, पीऽ) not Hindi forms |
| PM Kisan | Mentions **6000 रुपिया** and **तीन किस्त** |
| Sunrise | Answer says **पूरुब** (not पूर्व) |

## Next step
Replace the base model in `backend-api/app.py` with the fine-tuned adapter:
```python
from peft import PeftModel
base = AutoModelForCausalLM.from_pretrained(MODEL_ID, ...)
model = PeftModel.from_pretrained(base, ADAPTER_DIR)
```